## Session 5, in class: Markov chains with QuantEcon, time iteration, and the endogenous grid method

Computational Macro (HWS 2026), University of Mannheim. Thursday, 08.10.2026.

Three parts, one notebook. Part 1 puts income on a grid with the two methods of the slides, Tauchen and
Rouwenhorst, using the implementations of the `QuantEcon` package, and checks what the chains deliver.
Parts 2 and 3 solve the CARA household of the slides with the two Euler equation methods, time iteration
and the endogenous grid method, and measure both against the exact policy.

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top:
  later cells use what earlier cells define.
- There are **10 gaps**, marked `___`. The comment line `# Gap k: ...` directly above a gap says what goes in.
  Every gap is one line, and most are one line of the slides.
- Cells without a gap run as they are. Read them anyway: they are part of the method, and the comments
  explain each step.
- Each part ends with a **Check**: a number to compare with. Do not move on before it fits.
- **Slides.** "Slide 12" and the like refer to the session 5 slides of Wednesday, by the number at the
  bottom of each slide. Keep them open next to the notebook: most gaps are one line of a slide.
- Stuck on one gap for more than five minutes? Reread the hint and the slide, ask your neighbour, or ask us.
  The solution notebook is posted after the session.

**Plan**

| Part | Content | Gaps | Slides | Time |
| --- | --- | --- | --- | --- |
| 1 | Income on a grid: Tauchen and Rouwenhorst with `QuantEcon`, moments, simulation | 1 to 4 | 2 to 7 | 30 min |
| 2 | Time iteration on the CARA household: expected marginal utility, the residual, bisection | 5 to 7 | 8 to 14, 21 | 30 min |
| 3 | The endogenous grid method | 8 to 10 | 17 to 20, 22 | 20 min |

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $\rho$, $\sigma_\varepsilon$, $\sigma_z = \sigma_\varepsilon / \sqrt{1 - \rho^2}$ | persistence, innovation and unconditional standard deviation of $\ln z$ | `rho`, `sig_eps`, `sig_z` |
| $\mathcal{Z}$, $\boldsymbol{\Pi}$, $\pi_{mn} = \Pr(z' = z_n \mid z = z_m)$ | income grid and transition matrix, rows sum to one | `zgrid`, `Pi` |
| $\boldsymbol{\lambda}$, $\boldsymbol{\lambda} = \boldsymbol{\Pi}^\top \boldsymbol{\lambda}$ | stationary distribution | `lam` |
| $\mathcal{A}$, $x_{im} = z_m + R a_i$ | asset grid and cash on hand | `gri.a`, `gri.x` |
| $\mathbf{c}_n$ | consumption policy on the grid, iterate $n$ | `C` |
| $\mathbf{EMU} = u_c(\mathbf{c}_n) \boldsymbol{\Pi}^\top$ | expected marginal utility tomorrow, at every $(a'_l, z_m)$ | `EMU` |
| $f_{im}(c) = u_c(c) - \beta R \, \widehat{EMU}_m(x_{im} - c)$ | the time iteration residual | `resid(c)` |
| $c^{\min}_{im}, c^{\max}_{im}$ | the bracket from the grid limits | `cmin`, `cmax` |
| $\tilde c_{lm}, \tilde a_{lm}$ | the endogenous grid of one EGM step | `ctilde`, `atilde` |
| $e_W$ | largest error against the exact policy on $a \in [0, 8]$ | `err_window` |

**Toolbox: the few NumPy tools this notebook uses**

| Tool | What it does | Example |
| --- | --- | --- |
| `A @ B` | matrix product (also vector times matrix) | `lam @ zgrid` is $\sum_m \lambda_m z_m$ |
| `A.sum(axis=1)` | sum along each row | `Pi.sum(axis=1)` gives one number per row |
| `x[:, None]`, `x[None, :]` | turn a vector into a column, or into a row | `a[:, None] + z[None, :]` is the matrix of all sums $a_i + z_m$ |
| `np.outer(x, y)` | the matrix with entries $x_m y_n$ | |
| `np.where(cond, A, B)` | elementwise: `A` where `cond` is `True`, `B` elsewhere | |
| `np.interp(xq, x, f)` | linear interpolation of the points `(x, f)` at the query points `xq` | |
| `np.searchsorted(row, u)` | the first index `n` with `row[n] >= u` (for a sorted `row`) | |

Python counts from 0: the first entry of `x` is `x[0]`, the last is `x[-1]`. A policy is stored as an
$N_a \times N_z$ array: row `i` is the asset grid point $a_i$, column `m` the income state $z_m$.

### 0. Packages

New today: `quantecon`, the Python package behind the QuantEcon lectures. It brings `tauchen`, `rouwenhorst`,
and a `MarkovChain` class with a simulator and the stationary distribution. The first cell installs it
into the course environment if it is missing. The `warnings` line silences a notice that `rouwenhorst`
prints on every call about a change of its argument order in an old version (we use the current one).

In [ ]:
# Installs the packages of this notebook into the environment of the running kernel.
%pip install "numpy==2.2.*" "matplotlib==3.10.*" "quantecon>=0.7"

In [ ]:
import time
import warnings
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
import quantecon as qe

warnings.filterwarnings("ignore", message="The API of rouwenhorst")   # the notice on every rouwenhorst call, see above

### 1. Income on a grid (gaps 1 to 4)

*Session 5, slides 2 to 7.*

Income is persistent in logs, $\ln z' = \rho \ln z + \sigma_\varepsilon \varepsilon'$ with
$\varepsilon' \sim N(0, 1)$, as in session 4. We replace this continuous process by a Markov chain:
$N_z$ grid points for $\ln z$ and a matrix $\boldsymbol{\Pi}$ of transition probabilities. A good chain
matches three numbers of the process: its mean $0$, its unconditional standard deviation
$\sigma_z = \sigma_\varepsilon / \sqrt{1 - \rho^2}$, and its autocorrelation $\rho$.

In [ ]:
# The income process of session 4: ln z' = rho ln z + sig_eps eps',  eps' ~ N(0, 1)
rho = 0.9          # persistence
sig_eps = 0.2      # standard deviation of the innovation
nz = 7             # number of grid points for ln z, N_z

# The unconditional standard deviation of ln z: one of the three targets for the chain
sig_z = sig_eps / np.sqrt(1 - rho ** 2)         # unconditional sd of an AR(1): sig_eps / sqrt(1 - rho^2)
print(f"targets: mean 0, standard deviation {sig_z:.4f}, autocorrelation {rho}")

**Tauchen (1986), slide 4.** The function call

`mc = qe.tauchen(n, rho, sigma, mu=0.0, n_std=3)`

builds the chain for
$y' = \mu + \rho y + \varepsilon'$ with $\varepsilon' \sim N(0, \sigma^2)$: `sigma` is the standard deviation
of the **innovation**, `n_std` is the $m$ of the slides (the grid runs over $\pm m \sigma_z$). It returns a
`MarkovChain` object with the transition matrix in `.P` and the grid in `.state_values`.

In [ ]:
# Build the Tauchen chain for ln z (slide 4): N_z grid points and the N_z x N_z transition matrix.
# The result mc_t bundles both, the lines after the call print them.
# Gap 1: call qe.tauchen with the number of states nz, the persistence rho, the innovation standard deviation
#        sig_eps, and n_std=3 (the m of the slides)
mc_t = qe.tauchen(nz, rho, sig_eps, n_std=3)

# mc_t is a MarkovChain object with two pieces:
#   mc_t.state_values   the grid for ln z, N_z points
#   mc_t.P              the transition matrix, N_z x N_z: row m holds the probabilities of tomorrow's states
#                       given state m today
np.set_printoptions(precision=3, suppress=True, linewidth=120)   # print 3 digits, no scientific notation
print("grid for ln z:", mc_t.state_values)
print("transition matrix, rows are today, columns tomorrow:")
print(mc_t.P)

**Rouwenhorst (1995), slides 5 and 6.** The function call

`mc = qe.rouwenhorst(n, rho, sigma, mu=0.0)`

takes the same arguments as `qe.tauchen` without `n_std`: `sigma` is again the standard deviation of the
**innovation**. The grid runs over $\pm \sqrt{N_z - 1} \, \sigma_z$, and the matrix is built from the two
state chain by padding, as on the slides. It returns a `MarkovChain` object, as `qe.tauchen` does.

In [ ]:
# Build the Rouwenhorst chain with the same arguments (slides 5 and 6), then check that every row of
# its transition matrix is a probability distribution.
# Gap 2: call qe.rouwenhorst with the number of states, rho, and the innovation standard deviation
mc_r = qe.rouwenhorst(nz, rho, sig_eps)
print("grid for ln z:", mc_r.state_values)
print("transition matrix:")
print(mc_r.P)

In [ ]:
# Every row of P is a probability distribution over tomorrow's states, so every row must sum to one
# Gap 3: the sum of each row of mc_r.P, that is along axis 1
rowsums = mc_r.P.sum(axis=1)
print("rows sum to one:", np.allclose(rowsums, 1))

mid_state = nz // 2                                # index of the middle state (Python counts from 0)
print("probability to stay in the middle state:", round(mc_r.P[mid_state, mid_state], 3))

**What the chain delivers.** Three numbers, computed from the grid and the matrix alone.

- The stationary distribution $\boldsymbol{\lambda}$ solves $\boldsymbol{\lambda} = \boldsymbol{\Pi}^\top \boldsymbol{\lambda}$:
  it is the eigenvector of $\boldsymbol{\Pi}^\top$ to the eigenvalue one (slide 7). Here it is only a tool
  for the moments; next week is about distributions in their own right.
- Mean and variance weigh the grid with it: $\bar z = \sum_m \lambda_m z_m$ and
  $\text{Var}(z) = \sum_m \lambda_m (z_m - \bar z)^2$.
- The autocorrelation needs the probability of each pair (today, tomorrow): the pair $(z_m, z_n)$ has
  probability $\lambda_m \pi_{mn}$, so $\text{Cov}(z, z') = \sum_m \sum_n \lambda_m \pi_{mn} (z_m - \bar z)(z_n - \bar z)$,
  and the autocorrelation is $\text{Cov}(z, z') / \text{Var}(z)$.

In [ ]:
# The two functions below turn the formulas of the slides into code: the stationary distribution lam
# (slide 7), and the mean, standard deviation, and autocorrelation that Tauchen and Rouwenhorst are built
# to match (slides 3 to 6). The formulas are the three bullets above.
def stationary(Pi):
    # The stationary distribution lam of the chain: lam = Pi^T lam, so lam is the eigenvector of Pi^T
    # to the eigenvalue one, scaled to sum to one.
    eigenvalues, eigenvectors = np.linalg.eig(Pi.T)  # all eigenvalues and eigenvectors of Pi^T
    k = np.argmin(np.abs(eigenvalues - 1))      # position of the eigenvalue closest to one
    lam = np.real(eigenvectors[:, k])           # its eigenvector (eig returns complex numbers, the imaginary part is 0)
    return lam / lam.sum()                      # an eigenvector has any scale: normalize it to a distribution

In [ ]:
# The moments of the chain, with the formulas of the bullets above: weigh the grid with lam.
def chain_moments(zgrid, Pi):
    # Mean, standard deviation, and autocorrelation of the chain (zgrid, Pi) in its stationary distribution
    lam = stationary(Pi)                        # the weights of the grid points in the long run
    mean = lam @ zgrid                          # sum_m lam_m z_m
    dev = zgrid - mean                          # deviation from the mean, one per state
    var = lam @ dev ** 2                        # sum_m lam_m (z_m - mean)^2
    joint = lam[:, None] * Pi                   # joint[m, n] = lam_m Pi_mn, the probability of the pair (z_m, z_n)
    # Cov(z, z') = sum over all pairs (m, n) of joint[m, n] dev[m] dev[n]. np.outer(dev, dev) is the matrix
    # of the products dev[m] dev[n]: multiply it elementwise with joint and add up all entries.
    cov = np.sum(joint * np.outer(dev, dev))
    return mean, np.sqrt(var), cov / var        # the autocorrelation is Cov(z, z') / Var(z)

In [ ]:
# Apply chain_moments to both chains and compare with the targets printed at the top of part 1.
for name, mc in [("Tauchen", mc_t), ("Rouwenhorst", mc_r)]:
    mean, sd, ac = chain_moments(mc.state_values, mc.P)  # the three moments of this chain
    print(f"{name:12s} mean {mean:+.4f}  standard deviation {sd:.4f}  autocorrelation {ac:.4f}")

Rouwenhorst hits both targets exactly, Tauchen does not: with $\rho = 0.9$ and seven points the
standard deviation is $0.54$ instead of $0.46$. The cell below repeats the comparison for three
persistences (nothing to fill in). Watch what happens to Tauchen's autocorrelation at $\rho = 0.99$, and
compare with Giovanni's presentation from yesterday.

In [ ]:
# Nothing to fill in. The same comparison for three persistences rho_ = 0.5, 0.9, 0.99. For each rho_:
#   1. compute the target standard deviation of the AR(1), sig_eps / sqrt(1 - rho_^2),
#   2. build both chains with this rho_ (the same calls as above),
#   3. compute their moments with chain_moments and print one row per method.
# Columns: persistence, method, standard deviation of the chain, its target, autocorrelation of the
# chain (its target is rho_ itself).
print(f"{'rho':>5s} {'method':12s} {'sd':>8s} {'target':>8s} {'autocorr':>9s}")
for rho_ in [0.5, 0.9, 0.99]:
    target = sig_eps / np.sqrt(1 - rho_ ** 2)   # the standard deviation the chain should have
    chains = [("Tauchen", qe.tauchen(nz, rho_, sig_eps, n_std=3)),
              ("Rouwenhorst", qe.rouwenhorst(nz, rho_, sig_eps))]
    for name, mc in chains:   # the two chains for this rho_, one row each
        _, sd, ac = chain_moments(mc.state_values, mc.P)  # the mean is zero for both, so we skip it (_)
        print(f"{rho_:5.2f} {name:12s} {sd:8.4f} {target:8.4f} {ac:9.4f}")

**Simulation.** One draw per period, as on slide 7: cumulate each row once, draw $u \sim U(0, 1)$,
and move to the first state $n$ whose cumulated probability reaches $u$. Example: if the row of today's
state is $(0.2, 0.5, 0.3)$, the cumulated row is $(0.2, 0.7, 1.0)$, and $u = 0.45$ moves the chain to the
second state. The function below does it by hand.
`QuantEcon` does the same in `mc.simulate_indices(T, init=..., random_state=...)` (`mc.simulate` returns
the grid values instead of the indices). The seed makes the draws reproducible: same seed, same path.

In [ ]:
# Simulation by hand (slide 7): cumulate each row once, then one uniform draw per period picks
# tomorrow's state from the cumulated row of today's state.
def simulate_chain(Pi, T, m0, seed):
    # A simulated path of T state indices, starting in state m0
    rng = np.random.default_rng(seed)           # random number generator, the seed makes the path reproducible
    F = np.cumsum(Pi, axis=1)                   # cumulate each row once: F[m, n] = Pi[m, 0] + ... + Pi[m, n]
    s = np.empty(T, dtype=int)                  # s[t] is the index of the state in period t
    s[0] = m0                                   # the path starts in state m0
    for t in range(T - 1):
        u = rng.random()                        # a fresh uniform draw on (0, 1)
        # Gap 4: the first n with F[s[t], n] >= u: np.searchsorted(row, u) returns it, the row is today's
        #        cumulated row F[s[t]]
        s[t + 1] = np.searchsorted(F[s[t]], u)
    return s

In [ ]:
T = 100_000                                     # number of simulated periods (the _ only makes the number readable)
s = simulate_chain(mc_r.P, T, m0=nz // 2, seed=11)      # start in the middle state
zpath = mc_r.state_values[s]                            # from state indices to values of ln z


def autocorr(x):
    # correlation between x_t and x_{t+1}
    return np.corrcoef(x[:-1], x[1:])[0, 1]     # x[:-1] is x_0..x_{T-2}, x[1:] is x_1..x_{T-1}: today against tomorrow


print(f"by hand:    standard deviation {zpath.std():.4f}, autocorrelation {autocorr(zpath):.4f}")

In [ ]:
# The same with the simulator of QuantEcon
s_qe = mc_r.simulate_indices(T, init=nz // 2, random_state=11)  # QuantEcon's simulator: T state indices, same start, same seed
zpath_qe = mc_r.state_values[s_qe]              # from state indices to values of ln z
print(f"QuantEcon:  standard deviation {zpath_qe.std():.4f}, autocorrelation {autocorr(zpath_qe):.4f}")

In [ ]:
# Nothing to fill in: the first 120 periods of the simulated path
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.step(np.arange(120), zpath[:120], where="post")
for zi in mc_r.state_values:
    ax.axhline(zi, color="gray", lw=0.5, ls=":")               # the grid points as dotted lines
ax.set(title="A simulated chain, 120 periods", xlabel="period t", ylabel="ln z")
plt.tight_layout()
plt.show()

**Check:** by hand and with `QuantEcon`, the simulated standard deviation is about $0.46$ and the
autocorrelation about $0.90$. If your numbers are far off, look at gap 4: the cumulated row and the `searchsorted`.

### 2. Time iteration on the CARA household (gaps 5 to 7)

*Session 5, slides 8 to 14, the benchmark on slide 21.*

The household of slides 8 and 9: $u(c) = -e^{-\gamma c} / \gamma$, $c + a' = z + R a$, i.i.d. income
$z = \bar z + e$ with $e$ on $N_z = 7$ Rouwenhorst points. With $\rho = 0$ all rows of
$\boldsymbol{\Pi}$ are identical, the row is the weight vector $\omega$. Parameters as on the slides:
$\gamma = 2$, $r = 0.04$, $\beta = 1/R$, $\bar z = 1$, $\sigma = 0.2$, $N_a = 200$ equidistant points
on $[-6, 14]$. The bounds of the asset grid act as a borrowing and a saving limit.

**The benchmark.** The exact policy of the discretized problem is $c = \mu x + f_{N_z}$ with $\mu = r / R$ and

$$
f_{N_z} = \frac{\mu \bar z}{r} - \frac{1}{\gamma r} \Bigl[ \ln(\beta R) + \ln \sum_m \omega_m \exp\bigl(-\gamma \mu (z_m - \bar z)\bigr) \Bigr],
$$

slide 21, "On the computer: an exact benchmark". Errors are measured on the window $a \in [0, 8]$, six
units inside each bound, where the limits do not reach.

**Time iteration in one sentence (slides 11 and 12).** Guess a policy $\mathbf{c}_n$, and at every grid point find the $c$ that
solves the Euler equation when tomorrow's consumption follows the guess. That gives $\mathbf{c}_{n+1}$.
Repeat until the policy stops changing. The next cells build this one step at a time.

The first two cells set up parameters, grids, and the benchmark. Nothing to fill in, but read the comments.

In [ ]:
@dataclass(frozen=True)
class EconomicParameters:
    gamma: float = 2.0        # absolute risk aversion
    r: float = 0.04           # net interest rate, R = 1 + r
    beta: float = 1 / 1.04    # discount factor, beta R = 1
    zbar: float = 1.0         # mean income
    sigma: float = 0.2        # unconditional standard deviation of income
    rho: float = 0.0          # persistence of income, 0 is i.i.d.

    @property
    def R(self):              # gross interest rate, use it as par.R
        return 1 + self.r                       # R = 1 + r


@dataclass(frozen=True)
class NumericalParameters:
    na: int = 200             # asset grid points N_a
    nz: int = 7               # income grid points N_z
    amin: float = -6.0        # lower grid bound a_1
    amax: float = 14.0        # upper grid bound a_{N_a}
    crit_pol: float = 1e-6    # tolerance on the policy, both methods stop when the policy moves by less
    maxiter: int = 5000       # iteration cap, the safety net
    nbisect: int = 60         # bisection steps per time iteration step: bracket width 20 / 2^60
    window: float = 6.0       # errors are reported on [a_1 + window, a_N - window]

In [ ]:
@dataclass(frozen=True)
class Grids:
    a: np.ndarray             # asset grid, N_a
    z: np.ndarray             # income grid, N_z
    Pi: np.ndarray            # transition matrix, N_z x N_z
    x: np.ndarray             # cash on hand x_im = z_m + R a_i, N_a x N_z


def make_grids(par, mpar):
    # Income z = zbar + e, with e on a Rouwenhorst grid. QuantEcon takes the standard deviation of the
    # innovation, sd(e) * sqrt(1 - rho^2), so that the standard deviation of e itself is par.sigma.
    sig_innov = par.sigma * np.sqrt(1 - par.rho ** 2)  # innovation sd that gives sd(e) = sigma
    mc = qe.rouwenhorst(mpar.nz, par.rho, sig_innov)  # the income chain, as in part 1
    a = np.linspace(mpar.amin, mpar.amax, mpar.na)       # N_a equidistant asset grid points
    z = par.zbar + mc.state_values                       # income levels
    # Cash on hand x_im = z_m + R a_i for all combinations: a[:, None] is a column (assets down the rows),
    # z[None, :] is a row (income along the columns), and their sum is an N_a x N_z matrix.
    x = z[None, :] + par.R * a[:, None]         # N_a x N_z: row i is a_i, column m is z_m
    return Grids(a=a, z=z, Pi=mc.P, x=x)        # bundle everything in one container

In [ ]:
par, mpar = EconomicParameters(), NumericalParameters()  # the default values listed above
gri = make_grids(par, mpar)                     # grids, transition matrix, cash on hand
print("income grid:", gri.z)
print("weights:    ", gri.Pi[0], " all rows identical:", np.allclose(gri.Pi, gri.Pi[0]))
print("shape of cash on hand:", gri.x.shape, "= (N_a, N_z)")

In [ ]:
def uprime(c, par):
    # marginal utility of u(c) = -exp(-gamma c) / gamma, that is u'(c) = exp(-gamma c)
    return np.exp(-par.gamma * c)               # u'(c) = exp(-gamma c), works on numbers and arrays


def uprime_inv(m, par):
    # inverse of marginal utility: the c with u'(c) = m (needed in part 3)
    return -np.log(m) / par.gamma               # solves exp(-gamma c) = m for c

In [ ]:
def c_exact(par, gri):
    # The benchmark: the exact policy c = mu x + f of the discretized i.i.d. problem, from the slides
    R, r = par.R, par.r                         # short names for the formula
    mu = r / R                                  # marginal propensity to consume out of cash on hand
    omega = gri.Pi[0]                           # weights of the income states (all rows of Pi are the same)
    e = gri.z - par.zbar                        # income shocks
    risk_term = np.log(np.sum(omega * np.exp(-par.gamma * mu * e)))  # the precautionary term of the formula
    f = mu * par.zbar / r - (np.log(par.beta * R) + risk_term) / (par.gamma * r)  # the intercept f_{N_z} of the slide
    return mu * gri.x + f                       # c = mu x + f at every grid point


def err_window(C, par, mpar, gri):
    # Largest absolute consumption error against the exact policy: on the window, and on the whole grid
    err = np.abs(C - c_exact(par, gri))         # absolute error at every grid point
    inside = (gri.a >= gri.a[0] + mpar.window) & (gri.a <= gri.a[-1] - mpar.window)   # rows in the window
    return err[inside].max(), err.max()         # largest error on the window, and on the whole grid

In [ ]:
mu = par.r / par.R                              # the marginal propensity to consume out of cash on hand
f = c_exact(par, gri)[0, 0] - mu * gri.x[0, 0]  # read the intercept off the exact policy
print(f"exact policy: mu = {mu:.5f}, intercept f = {f:.5f}")

The starting policy is the closed form without its precautionary term, $c_0 = \mu x + (1 - \mu) \bar z$,
the policy of a household that ignores the risk (the slides' $c = r a + \bar z$ works as well, it costs one
iteration more).

**Step 2 of the algorithm on slide 12: expected marginal utility (gap 5).** Given the guess $\mathbf{c}_n$ on the grid,
tomorrow's expected marginal utility at every $(a'_l, z_m)$ is one matrix product,
$\mathbf{EMU} = u_c(\mathbf{c}_n) \boldsymbol{\Pi}^\top$, exactly as $\mathbf{EV} = \mathbf{V} \boldsymbol{\Pi}^\top$ in week 4:
$EMU_{lm} = \sum_n \pi_{mn} \, u_c(c_n(a'_l, z_n))$. In words: a household with income $z_m$ today
draws tomorrow's income from row $m$ of $\boldsymbol{\Pi}$, and weighs tomorrow's marginal utilities with it.

In [ ]:
# Step 2 of time iteration (slide 12): from the guess C to expected marginal utility tomorrow, at every
# grid point for tomorrow's assets and every income state today.
def expected_mu(C, par, gri):
    # EMU[l, m] = sum_n Pi[m, n] u'(C[l, n]): expected marginal utility tomorrow for a household that
    # saves a'_l and has income z_m today. Input C and output EMU are both N_a x N_z.
    # Gap 5: one matrix product, as EV = V Pi^T in week 4, with the marginal utilities uprime(C, par) in place
    #        of V
    return uprime(C, par) @ gri.Pi.T

In [ ]:
# Test: with c = 1 everywhere, marginal utility is exp(-gamma) everywhere, and so is its expectation
EMU = expected_mu(np.ones_like(gri.x), par, gri)  # C = 1 at every grid point
print("expected marginal utility test:", np.allclose(EMU, np.exp(-par.gamma)))

**Step 3 on slide 12: the residual (gap 6).** For every $(a_i, z_m)$ and a candidate $c$, tomorrow's assets are
$a' = x_{im} - c$, in general off the grid, so column $m$ of $\mathbf{EMU}$ is interpolated linearly at $a'$:

$$
f_{im}(c) = u_c(c) - \beta R \, \widehat{EMU}_m(x_{im} - c).
$$

`resid` takes the whole $N_a \times N_z$ array of candidates at once and returns the array of residuals, one
interpolation per income state. It is decreasing in $c$ and has exactly one root per point:
$f > 0$ means $c$ is too low (marginal utility today is too high), $f < 0$ means $c$ is too high.

In [ ]:
# Step 3 (slide 12): the Euler equation residual, in three moves: tomorrow's assets from the budget
# constraint, expected marginal utility at those assets (interpolated, gap 6), and u'(c) - beta R EMUhat.
def resid(c, EMU, par, gri):
    # The Euler equation residual f_im(c) = u'(c) - beta R EMUhat_m(x_im - c), for an N_a x N_z array c
    aprime = gri.x - c                          # tomorrow's assets implied by c, in general off the grid
    EMUhat = np.empty_like(c)                   # EMU interpolated at aprime, filled one column at a time
    for m in range(len(gri.z)):                 # one income state at a time
        # Gap 6: interpolate column m of EMU (values on the grid gri.a) at the choices aprime[:, m]:
        #        np.interp(query points, grid, values)
        EMUhat[:, m] = np.interp(aprime[:, m], gri.a, EMU[:, m])
    return uprime(c, par) - par.beta * par.R * EMUhat  # f = u'(c) - beta R EMUhat: zero at the optimal c

In [ ]:
def guess(par, gri):
    # The starting policy c0 = mu x + (1 - mu) zbar: the closed form without the precautionary term.
    # np.minimum caps it at x - a_1, the most a household can consume without borrowing beyond a_1.
    mu = par.r / par.R                          # the marginal propensity to consume out of cash on hand
    c0 = mu * gri.x + (1 - mu) * par.zbar       # consume the annuity value of cash on hand and income
    return np.minimum(c0, gri.x - gri.a[0])     # never consume so much that a' < a_1

In [ ]:
C0 = guess(par, gri)                            # the starting policy
f0 = resid(C0, expected_mu(C0, par, gri), par, gri)  # how far the guess is from solving the Euler equation
print(f"residual at the guess, largest absolute value: {np.max(np.abs(f0)):.4f}  (zero would mean the guess is the fixed point)")

**Steps 4 and 5: the limits, and bisection on all points at once (gap 7).** Bisection is on slides 13 and 14,
the bracket and the vectorized version on the backup slides "Bounds on consumption" and "Many points at once". The bracket comes from the grid,
$c^{\max} = x_{im} - a_1$ and $c^{\min} = x_{im} - a_{N_a}$. If $f_{im}(c^{\max}) \geq 0$ the household wants to
borrow beyond $a_1$ and the limit binds, if $f_{im}(c^{\min}) \leq 0$ it wants to save beyond $a_{N_a}$.
Otherwise the root lies inside, and bisection halves every bracket $60$ times: since $f$ is decreasing,
a positive residual at the midpoint means the root is to the right, so the lower end moves up to the
midpoint; otherwise the upper end moves down. The update of the lower end is given, gap 7 is the upper end.

In [ ]:
# Step 5 (slides 13 and 14, backup slide "Many points at once"): bisection on all grid points at once.
# Each pass halves every bracket [lo, hi]: evaluate f at the midpoint and keep the half with the root.
def bisect_vec(f, lo, hi, steps):
    # Bisection on every grid point at once (backup slide "Many points at once"): lo and hi are N_a x N_z arrays,
    # and the root of the decreasing function f lies between them, point by point.
    for _ in range(steps):
        mid = (lo + hi) / 2                     # midpoint of every bracket
        pos = f(mid) > 0                        # True where f(mid) > 0: the root is in [mid, hi]
        lo = np.where(pos, mid, lo)             # where pos, the lower end moves up to mid, elsewhere it stays
        # Gap 7: the upper end: where pos it stays at hi, elsewhere (root in [lo, mid]) it moves down to mid.
        #        Mirror the line above
        hi = np.where(pos, hi, mid)
    return (lo + hi) / 2                        # after the last step, the midpoint is the root (up to 20 / 2^60)

In [ ]:
def time_iteration_step(C, par, mpar, gri):
    # One step of time iteration (the Coleman operator): the new policy from the guess C
    EMU = expected_mu(C, par, gri)              # step 2: tomorrow's expected marginal utility under the guess
    f = lambda c: resid(c, EMU, par, gri)       # step 3: the residual as a function of c alone

    # Step 4: the bracket from the grid limits, and where a limit binds
    cmax = gri.x - gri.a[0]                     # consume everything down to a' = a_1
    cmin = gri.x - gri.a[-1]                    # save everything up to a' = a_N
    borrow = f(cmax) >= 0                       # even cmax is too little: the borrowing limit binds
    save = f(cmin) <= 0                         # even cmin is too much: the saving limit binds

    # Step 5: the root by bisection, then the limits where they bind
    c_root = bisect_vec(f, cmin, cmax, mpar.nbisect)  # the root of f in [cmin, cmax], at every grid point
    Cnew = np.where(save, cmin, c_root)         # where the saving limit binds, consume cmin
    Cnew = np.where(borrow, cmax, Cnew)         # where the borrowing limit binds, consume cmax
    return Cnew                                 # the new policy

In [ ]:
C1 = time_iteration_step(C0, par, mpar, gri)    # one step from the guess
interior = (C1 < gri.x - gri.a[0]) & (C1 > gri.x - gri.a[-1])      # points where neither limit binds
max_move = np.max(np.abs(C1 - C0))              # how much one step changes the policy
res = np.max(np.abs(resid(C1, expected_mu(C0, par, gri), par, gri)[interior]))  # is C1 the root, given the EMU of the guess?
print(f"one step moves the guess by at most {max_move:.4f}")
print(f"residual after the step, where no limit binds: {res:.1e}  (machine precision: bisection found the roots)")

**Step 6: the loop.** Apply the step until the policy moves by less than $\varepsilon_{\text{pol}} = 10^{-6}$.
Nothing to fill in.

In [ ]:
def time_iteration(par, mpar, gri):
    # Iterate time_iteration_step until the policy moves by less than crit_pol
    t0 = time.perf_counter()                    # start the clock
    C = guess(par, gri)                         # start from the guess
    dist = np.inf                               # distance between two iterates, infinite to enter the loop
    dists = []                                  # all distances, to look at the convergence afterwards
    it = 0                                      # iteration counter
    while dist > mpar.crit_pol and it < mpar.maxiter:
        Cnew = time_iteration_step(C, par, mpar, gri)  # one step of time iteration
        dist = np.max(np.abs(Cnew - C))         # largest change of the policy anywhere on the grid
        dists.append(dist)                      # remember the distance
        C = Cnew                                # the new policy is the guess for the next step
        it += 1                                 # count the iteration
    eW, eG = err_window(C, par, mpar, gri)      # errors against the exact policy
    return dict(C=C, iters=it, time=time.perf_counter() - t0, dist=dists,
                converged=dist <= mpar.crit_pol, eW=eW, eG=eG)

In [ ]:
ti = time_iteration(par, mpar, gri)             # solve the household problem with time iteration
print(f"time iteration: {ti['iters']} iterations, converged {ti['converged']}, {ti['time']:.2f} s, "
      f"{1000 * ti['time'] / ti['iters']:.1f} ms per iteration, error on the window {ti['eW']:.1e}, on the grid {ti['eG']:.1e}")

**Check:** 121 iterations and an error on the window of $6 \cdot 10^{-5}$, the numbers of slide 22,
"Four methods against the exact policy". The error on the whole grid is $0.47$: that is the limit at the
bottom of the grid, where the two lowest income states would like to borrow more, not a bug. The time per
iteration is what the endogenous grid method removes. If the loop does not converge, check gap 7 first:
with a wrong bracket update bisection finds no root.

### 3. The endogenous grid method (gaps 8 to 10)

*Session 5, slides 17 to 20.*

Carroll's trick (slide 17): fix tomorrow's assets $a'_l$ on the grid instead of today's. Then the right hand side of the
Euler equation is a number, $\beta R \, EMU_{lm}$, and consumption today follows by inverting marginal
utility, no root needed. The budget constraint then says where the household that chose $a'_l$ started:

$$
\tilde c_{lm} = u_c^{-1}\bigl( \beta R \, EMU_{lm} \bigr), \qquad \tilde a_{lm} = \frac{\tilde c_{lm} + a'_l - z_m}{R}.
$$

The points $(\tilde a_{lm}, \tilde c_{lm})$ lie on the new policy, on an endogenous grid that differs by
income state. Interpolating them back onto $\mathcal{A}$, one column at a time, gives $\mathbf{c}_{n+1}$ (slide 18).
As on slide 19, below $\tilde a_{1m}$ the borrowing limit binds and $c = x_{im} - a_1$, above $\tilde a_{N_a m}$ the saving
limit binds and $c = x_{im} - a_{N_a}$.

Compare with time iteration: steps 1 and 2 are the same (a guess and $\mathbf{EMU}$), only the step from
$\mathbf{EMU}$ to the new policy changes. The loop `egm` is the loop of part 2 with `egm_step` in place of
`time_iteration_step`.

In [ ]:
# One EGM step (algorithm on slide 20) in three moves: (a) consumption from the inverted Euler equation,
# (b) today's assets from the budget constraint (both slide 17), (c) back to the grid, with the limits
# where the endogenous grid ends (slides 18 and 19).
def egm_step(C, par, mpar, gri):
    # One step of the endogenous grid method: the new policy on the grid, and the endogenous grid of this step
    EMU = expected_mu(C, par, gri)              # as in time iteration: EMU[l, m] at tomorrow's assets a'_l and income z_m

    # (a) Consumption today from the Euler equation u'(ctilde) = beta R EMU, no root finding needed
    # Gap 8: invert marginal utility with uprime_inv, applied to beta R EMU (all points at once)
    ctilde = uprime_inv(par.beta * par.R * EMU, par)

    # (b) Where did the household start? The budget constraint c + a' = z + R a, solved for today's a
    aprime = gri.a[:, None]                     # tomorrow's assets a'_l as a column, N_a x 1
    z = gri.z[None, :]                          # income z_m as a row, 1 x N_z
    # Gap 9: today's assets a = (c + a' - z) / R, with ctilde as c. Column plus row gives an N_a x N_z matrix,
    #        like ctilde
    atilde = (ctilde + aprime - z) / par.R

    # (c) Back to the grid, one income state at a time
    Cnew = np.empty_like(C)                     # the new policy on the grid, filled one column at a time
    for m in range(len(gri.z)):
        # Gap 10: the points (atilde[:, m], ctilde[:, m]) lie on the new policy of income state m: interpolate
        #         them at the grid points gri.a, np.interp(query points, nodes, values)
        Cnew[:, m] = np.interp(gri.a, atilde[:, m], ctilde[:, m])
        # np.interp is flat outside the endogenous grid, so overwrite those points with the limits
        below = gri.a < atilde[0, m]            # poorer than the household that saves a' = a_1: borrowing limit binds
        above = gri.a > atilde[-1, m]           # richer than the household that saves a' = a_N: saving limit binds
        Cnew[below, m] = gri.x[below, m] - gri.a[0]  # consume everything down to a' = a_1
        Cnew[above, m] = gri.x[above, m] - gri.a[-1]  # save everything up to a' = a_N
    return Cnew, ctilde, atilde                 # the new policy, and the endogenous grid points

In [ ]:
def egm(par, mpar, gri):
    # The loop of time_iteration, with egm_step in place of time_iteration_step
    t0 = time.perf_counter()                    # start the clock
    C = guess(par, gri)                         # start from the guess
    dist = np.inf                               # infinite to enter the loop
    dists = []                                  # all distances
    it = 0                                      # iteration counter
    while dist > mpar.crit_pol and it < mpar.maxiter:
        Cnew, _, _ = egm_step(C, par, mpar, gri)   # egm_step also returns the endogenous grid, not needed here
        dist = np.max(np.abs(Cnew - C))         # largest change of the policy
        dists.append(dist)                      # remember the distance
        C = Cnew                                # the new policy is the guess for the next step
        it += 1                                 # count the iteration
    eW, eG = err_window(C, par, mpar, gri)      # errors against the exact policy
    return dict(C=C, iters=it, time=time.perf_counter() - t0, dist=dists,
                converged=dist <= mpar.crit_pol, eW=eW, eG=eG)

In [ ]:
eg = egm(par, mpar, gri)                        # solve the household problem with the endogenous grid method
print(f"endogenous grid: {eg['iters']} iterations, converged {eg['converged']}, {eg['time']:.3f} s, "
      f"{1000 * eg['time'] / eg['iters']:.2f} ms per iteration, error on the window {eg['eW']:.1e}, on the grid {eg['eG']:.1e}")

In [ ]:
# One more step from the converged policy, to look at its endogenous grid
_, ctilde, atilde = egm_step(eg["C"], par, mpar, gri)  # keep only the endogenous grid of this step
print("endogenous a at a' = a_1, by income state:", atilde[0])

**Check:** as on slide 22, 121 iterations, an error on the window of $2 \cdot 10^{-5}$, and the three lowest income
states have $\tilde a_{1m} > a_1 = -6$ (about $-5.7$, $-5.8$, $-6.0$), so the borrowing limit binds for them at
the bottom of the grid. The time per iteration should be an order of magnitude or two below time iteration.

In [ ]:
# Nothing to fill in. Left: one EGM step at the bottom of the grid for the lowest income state.
# Right: the error of both methods along the grid (largest over the income states), log scale.
ct = c_exact(par, gri)                          # the exact policy, for the comparison
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
sel = gri.a <= -2.0                             # the bottom of the grid, where the limit binds
ax[0].plot(atilde[:, 0], ctilde[:, 0], "o", ms=3, label="EGM points (atilde, ctilde), lowest income")
ax[0].plot(gri.a[sel], eg["C"][sel, 0], "k-", lw=1.2, label="policy on the grid")
ax[0].plot(gri.a[sel], ct[sel, 0], ":", lw=1.6, label="exact policy without the limit")
ax[0].axvline(atilde[0, 0], color="gray", ls="--", lw=1)     # below this point the borrowing limit binds
ax[0].set(xlim=(gri.a[0] - 0.1, -2.0), ylim=(eg["C"][0, 0] - 0.03, ct[sel, 0].max() + 0.03),
          xlabel="assets today a", ylabel="consumption", title="Back to the grid, and the limit")
ax[0].legend(fontsize=7)
for name, res in [("time iteration", ti), ("endogenous grid", eg)]:
    err_by_a = np.max(np.abs(res["C"] - ct), axis=1)          # largest error over the income states, per grid point
    ax[1].semilogy(gri.a, np.maximum(err_by_a, 1e-16), label=name)
ax[1].axvspan(0, 8, color="gray", alpha=0.15, lw=0)
ax[1].set(xlabel="assets a", ylabel="max over m of |c - exact|", title="Error against the exact policy (shaded: the window)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# The horse race of the slides, for the two Euler equation methods
print(f"{'method':18s} {'iterations':>10s} {'ms/iteration':>13s} {'seconds':>8s} {'e_W':>9s} {'e_G':>7s}")
for name, res in [("time iteration", ti), ("endogenous grid", eg)]:
    print(f"{name:18s} {res['iters']:10d} {1000 * res['time'] / res['iters']:13.2f} {res['time']:8.3f} {res['eW']:9.1e} {res['eG']:7.2f}")

**To discuss (slides 22 to 24).** Both methods iterate on the same operator, so they need the same number of
iterations (121 here, against 296 for value function iteration on the slides). The difference is the cost
per iteration: 60 bisection steps with 7 interpolations each, against 7 interpolations in total. The
remaining error of both methods on the window is neither stopping error nor interpolation error: tightening
`crit_pol` to $10^{-10}$ or doubling $N_a$ twice leaves the $2 \cdot 10^{-5}$ of the endogenous grid method
where it is (try it). It is the grid bounds: with $\sigma = 0.2$ the limits reach into the window, as the
summary on slide 24 says. Problem set 2 has $\sigma = 0.1$, where they do not.